# CWRU Bearing Fault Diagnosis — Exploratory Data Analysis

This notebook walks through:
1. Loading raw `.mat` files
2. Visualising raw waveforms and FFT spectra
3. Comparing time-domain statistics across fault classes
4. Running the preprocessing pipeline
5. Spot-checking extracted features

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from load_data   import load_all_files, records_to_dataframe
from preprocess  import segment_records, normalise_windows, split_dataset
from features    import extract_features, feature_names

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

## 1. Load raw data

In [ ]:
# Set verbose=False if you just want the records without printing
records = load_all_files(verbose=True)
df_meta = records_to_dataframe(records)
df_meta

## 2. Waveform visualisation

In [ ]:
FS = 12_000  # sampling frequency (Hz)
N_PLOT = 2048  # samples to show

fig, axes = plt.subplots(len(records), 1, figsize=(14, 2.5 * len(records)), sharex=False)
if len(records) == 1:
    axes = [axes]

for ax, rec in zip(axes, records):
    t = np.arange(N_PLOT) / FS
    ax.plot(t, rec['signal'][:N_PLOT], linewidth=0.5)
    ax.set_title(rec['label'], fontsize=9)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('g')

plt.suptitle('Raw Vibration Waveforms', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/figures/waveforms.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. FFT spectra

In [ ]:
fig, axes = plt.subplots(len(records), 1, figsize=(14, 2.5 * len(records)), sharex=False)
if len(records) == 1:
    axes = [axes]

for ax, rec in zip(axes, records):
    sig  = rec['signal'][:4096]
    n    = len(sig)
    fft  = np.abs(np.fft.rfft(sig * np.hanning(n))) / n
    freq = np.fft.rfftfreq(n, d=1.0/FS)
    ax.semilogy(freq, fft, linewidth=0.6)
    ax.set_title(rec['label'], fontsize=9)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Amplitude')
    ax.set_xlim(0, FS/2)

plt.suptitle('Amplitude Spectra', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/figures/spectra.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Class distribution after segmentation

In [ ]:
X, y, labels = segment_records(records, window_size=1024, overlap=0.5)
print(f'Total windows: {len(X):,}  |  Feature shape: {X.shape}')

unique, counts = np.unique(y, return_counts=True)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([labels[i] for i in unique], counts, color=sns.color_palette('tab10', len(unique)))
ax.set_xlabel('Class')
ax.set_ylabel('Window count')
ax.set_title('Class distribution after segmentation')
plt.tight_layout()
plt.show()

## 5. Feature extraction spot-check

In [ ]:
# Extract features for a small subset to inspect
from features import extract_feature_matrix

sample_idx = np.random.choice(len(X), size=500, replace=False)
X_feat = extract_feature_matrix(X[sample_idx], verbose=True)
feat_df = pd.DataFrame(X_feat, columns=feature_names())
feat_df['class'] = y[sample_idx]
feat_df.head()

In [ ]:
# Kurtosis distribution per class
fig, ax = plt.subplots(figsize=(8, 4))
for cls_id, cls_name in enumerate(labels):
    vals = feat_df.loc[feat_df['class'] == cls_id, 'td_kurtosis']
    ax.hist(vals, bins=30, alpha=0.6, label=cls_name)

ax.set_xlabel('Kurtosis')
ax.set_ylabel('Count')
ax.set_title('Kurtosis Distribution by Fault Class')
ax.legend()
plt.tight_layout()
plt.show()